# Week 3 · Day 3 — Graph RAG: global questions & hybrid routing

**Zylo Technologies · AI Engineering Internship**
Prepared for **Graph RAG Project Task**

| Part | Topic |
|---|---|
| 1 | The **second wall** — a global question traversal can't touch |
| 2 | **Communities** — Louvain finds the topics in your graph |
| 3 | **Community reports** — summarize each cluster at index time |
| 4 | **Global search** — map-reduce over the reports |
| 5 | **Local vs global** — same graph, two retrievers |
| 6 | The **router** — vector · local · global, chosen automatically |

**Stack:** Groq (llama-3.3-70b-versatile) · NetworkX (graph + Louvain) · Voyage AI (voyage-3 embeddings).

In [1]:
import os, re, json, time, requests
import networkx as nx
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
VOYAGE_API_KEY = os.getenv('VOYAGE_API_KEY')
assert GROQ_API_KEY and VOYAGE_API_KEY
print('API Keys loaded successfully.')

Ready. Graph nodes: 19, edges: 19


## Part 1 · Corpus & The Global Wall
Corpus with genuine cluster structure: Payments team, Search team, and Auth/Platform dependencies.

In [2]:
DOCS = {
    'teams.md': 'The Payments team owns the billing-service. Dana Okafor leads the Payments team. The Search team owns the search-service. Priya Nair leads the Search team.',
    'billing.md': 'The billing-service exposes the invoicing API. The billing-service runs on the ledger-db.',
    'search.md': 'The search-service exposes the search API. The search-service runs on the index-db.',
    'platform.md': 'The billing-service depends on the auth-service. The search-service depends on the auth-service. The auth-service runs on the shared-db.',
    'customerA.md': 'Acme Corp subscribes to the invoicing API. Acme Corp is on the enterprise plan.',
    'customerB.md': 'Globex subscribes to the search API. Globex is on the enterprise plan.',
    'incident1.md': 'On Monday the billing-service had an outage caused by the ledger-db running out of connections.',
    'incident2.md': 'On Tuesday the search-service had an outage caused by the auth-service timing out.',
    'incident3.md': 'On Wednesday the billing-service slowed down because the auth-service was degraded.',
}
print('Corpus initialized with', len(DOCS), 'documents.')

Corpus: 9 docs
  - LOOKUP: What plan is Acme Corp on?
  - MULTIHOP: Which customer is affected by the outage on the billing-service?
  - GLOBAL: What is the most common root cause of outages across all services?


## Part 2 · Louvain Community Detection
Detecting structural topic communities on the undirected graph.

In [3]:
UG = G.to_undirected()
communities = nx.community.louvain_communities(UG, seed=42)
communities = [c for c in sorted(communities, key=len, reverse=True) if len(c) > 1]
for i, c in enumerate(communities):
    print(f'  community {i}: {sorted(c)}')

Louvain found 4 communities:

  community 0: ['billing-service', 'connections', 'dana okafor', 'ledger-db', 'ledger-db running out of connections', 'monday', 'payments team']
  community 1: ['acme corp', 'enterprise plan', 'globex', 'invoicing api', 'search api']
  community 2: ['index-db', 'priya nair', 'search team', 'search-service', 'tuesday']
  community 3: ['auth-service', 'shared-db']


## Part 3 · Precomputed Community Reports
Summarizing each cluster at index time to produce a global index.

In [4]:
for i, rep in enumerate(reports):
    print(f'── community {i} report ─────────────────────────')
    print(rep, '\n')

── community 0 report ─────────────────────────
The billing-service cluster is a system component owned by the payments team, led by Dana Okafor, which exposes the invoicing API and depends on the auth-service. The cluster experienced an outage on Monday due to the ledger-db running out of connections. Additionally, the billing-service was also slowed down because of issues with the auth-service, which was another cause of the failure.

── community 1 report ─────────────────────────
This cluster is comprised of two services: the billing-service, which exposes the invoicing api, and the search-service, which exposes the search api. Both Acme Corp and Globex subscribe to these APIs, with Acme Corp using the invoicing api and Globex using the search api, and both are on the enterprise plan. There is no information available on failures or causes within this cluster.

── community 2 report ─────────────────────────
The search-service cluster is a system component that exposes the search A

## Part 4 · Map-Reduce Global Search
Mapping global query over community reports, then reducing to discover aggregated root cause.

In [5]:
reduced_answer, map_traces = global_search(GLOBAL_Q, show=True)
print('GLOBAL SEARCH ANSWER:\n', reduced_answer)

GLOBAL QUESTION: What is the most common root cause of outages across all services?

MAP TRACES:
  [map] community 0: * The billing-service cluster experienced an outage due to the ledger-db running out of connections, and was also slowed down by issues with the auth-service, which it depends on.
* The root cause of the outage was the ledger-db connection issue, while the auth-service issues were a contributing factor to the failure.
  [map] community 1: * The cluster includes two services: billing-service (invoicing api) and search-service (search api), used by Acme Corp and Globex.
* There is no information available on failures or causes within this cluster, making it impossible to determine the root cause of outages from this report.
  [map] community 2: * The search-service cluster experienced an outage due to a failure in the auth-service, indicating a dependency-related issue.
* The auth-service is identified as the root cause of the outage in the search-service cluster, suggest

## Part 5 · Local vs Global Comparison
Side-by-side performance of Vector RAG, Local Traversal, and Global Map-Reduce.

In [6]:
print('Vector RAG  :', vector_search(GLOBAL_Q))
print('Local Graph :', local_search(GLOBAL_Q))
print('Global Graph:', global_search(GLOBAL_Q)[0])

=== GLOBAL QUESTION ===
What is the most common root cause of outages across all services?

Vector RAG  : I don't know.

Local Graph : I don't know (no entities/relationships matched in graph).

Global Graph: The most common root cause of outages across all services is issues related to the auth-service, specifically its instability and performance problems, which have a ripple effect on dependent services such as the billing-service and search-service. Additionally, database connection issues, such as the ledger-db running out of connections, also appear to be a contributing factor to outages. The pattern that emerges across multiple clusters is that dependency-related issues, particularly with the auth-service, are a primary cause of outages, highlighting the need to address the auth-service's instability and improve its performance to prevent cascading failures in other services.


## Part 6 · Hybrid Router & Benchmark
Classifying questions into `vector`, `local`, or `global` and dispatching automatically.

In [7]:
test_questions = [
    ('What plan is Acme Corp on?', 'vector'),
    ('Who leads the Search team?', 'vector'),
    ('Which customer is affected by the outage on the billing-service?', 'local'),
    ('Which database does the service led by Priya Nair run on?', 'local'),
    ('What is the most common root cause of outages across all services?', 'global'),
    ('What platform service creates a shared vulnerability between the Payments and Search teams?', 'global')
]
for q, exp in test_questions:
    p, r = route(q)
    print(f'Q: {q} -> {p.upper()} (expected {exp.upper()})')

Test Set Benchmark:

Q: What plan is Acme Corp on?
   Predicted: LOCAL (Expected: VECTOR)
   Reasoning: The question is about a specific named entity, Acme Corp, and its connection to a plan, requiring a multi-hop lookup to determine the relationship.
   Answer: Acme Corp is on the enterprise plan....

Q: Who leads the Search team?
   Predicted: VECTOR (Expected: VECTOR)
   Reasoning: This question requires a simple fact lookup answerable from a single passage about the leader of the Search team.
   Answer: Priya Nair leads the Search team....

Q: Which customer is affected by the outage on the billing-service?
   Predicted: LOCAL (Expected: LOCAL)
   Reasoning: This question requires connecting specific entities, such as the customer and the billing-service, to determine the affected party.
   Answer: To find the customer affected by the outage on the billing-service, we need to follow the chain of facts. 

1. The billing-service has an outage on Mo...

Q: Which database does the serv